In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATH
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2003_2004"
    / "Labs"
)

xpt_file = DATA_DIR / "L10AM_C.xpt"
csv_file = DATA_DIR / "L10AM_C_corrected.csv"


# ============================================================
# 2. READ XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

print("=" * 70)
print("NHANES 2003-2004 L10AM_C XPT -> CSV")
print("=" * 70)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))
print("Variables:", df.columns.tolist())


# ============================================================
# 3. KNOWN TINY ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


# ============================================================
# 4. CHECK BEFORE CORRECTION
# ============================================================

tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)

zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)

total_tiny = int(
    tiny_counts.sum()
)

total_zero = int(
    zero_counts.sum()
)


print("\n--- BEFORE CORRECTION ---")

print(
    "True numeric zeros:",
    f"{total_zero:,}"
)

print(
    "Tiny values:",
    f"{total_tiny:,}"
)


if total_tiny > 0:

    print("\nTiny values by variable:")

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
    )


# ============================================================
# 5. RESTORE WTSAF2YR ZEROS ONLY
# ============================================================

#
# IMPORTANT:
#
# In this file, the tiny value appears ONLY in WTSAF2YR.
#
# CDC permits WTSAF2YR = 0.
#
# We therefore correct the known XPT artifact ONLY
# in WTSAF2YR.
#
# We do NOT change any laboratory measurement values.
#

if "WTSAF2YR" in df.columns:

    mask = (
        df["WTSAF2YR"]
        .eq(TINY_VALUE)
    )

    print(
        "\nWTSAF2YR values to restore to zero:",
        int(mask.sum())
    )

    df.loc[
        mask,
        "WTSAF2YR"
    ] = 0.0


# ============================================================
# 6. MAKE SEQN A CLEAN INTEGER IDENTIFIER
# ============================================================

if "SEQN" not in df.columns:

    raise ValueError(
        "SEQN not found."
    )


seqn_values = (
    df["SEQN"]
    .dropna()
)


# SEQN must contain only whole-number values

if not np.all(
    np.isclose(
        seqn_values,
        np.round(seqn_values)
    )
):

    raise ValueError(
        "SEQN contains unexpected decimal values."
    )


df["SEQN"] = (
    df["SEQN"]
    .round()
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Missing:",
    int(df["SEQN"].isna().sum())
)

print(
    "Duplicates:",
    int(df["SEQN"].duplicated().sum())
)

print(
    "dtype:",
    df["SEQN"].dtype
)


# ============================================================
# 7. DO NOT CHANGE LAB VARIABLES
# ============================================================

#
# These are true continuous measurements.
# Keep their decimals exactly as read.
#

LAB_MEASUREMENTS = [
    "LBXGLU",
    "LBDGLUSI",
    "LBXCPSI",
    "LBXIN",
    "LBDINSI"
]


for col in LAB_MEASUREMENTS:

    if col in df.columns:

        print(
            f"{col}: dtype={df[col].dtype}, "
            f"min={df[col].min()}, "
            f"max={df[col].max()}, "
            f"missing={int(df[col].isna().sum())}"
        )


# ============================================================
# 8. CDC CODEBOOK VALIDATION
# ============================================================

print("\n--- CDC CODEBOOK CHECK ---")


# ------------------------------------------------------------
# TOTAL ROW COUNT
# ------------------------------------------------------------

if len(df) == 3356:

    print(
        "PASS: Row count = 3,356"
    )

else:

    print(
        "FAIL: Row count =",
        len(df)
    )


# ------------------------------------------------------------
# WTSAF2YR
# CDC:
# range 0 to 355659.48
# 3356 valid
# 0 missing
# ------------------------------------------------------------

if "WTSAF2YR" in df.columns:

    col = df["WTSAF2YR"]

    print("\nWTSAF2YR")

    print(
        "Zero count:",
        int(col.eq(0).sum())
    )

    print(
        "Missing:",
        int(col.isna().sum())
    )

    print(
        "Minimum:",
        col.min()
    )

    print(
        "Maximum:",
        col.max()
    )

    bad = (
        col.notna()
        &
        (
            (col < 0)
            |
            (col > 355659.48)
        )
    )

    if int(bad.sum()) == 0:

        print(
            "PASS: WTSAF2YR within CDC range."
        )

    else:

        print(
            "FAIL: WTSAF2YR outside CDC range:",
            int(bad.sum())
        )


# ------------------------------------------------------------
# LBXGLU
# CDC:
# 45.7 to 547.6
# valid 3169
# missing 187
# ------------------------------------------------------------

if "LBXGLU" in df.columns:

    col = df["LBXGLU"]

    valid_range = (
        col.between(
            45.7,
            547.6
        )
    )

    print("\nLBXGLU")

    print(
        "Valid:",
        int(valid_range.sum())
    )

    print(
        "Missing:",
        int(col.isna().sum())
    )

    if (
        int(valid_range.sum()) == 3169
        and int(col.isna().sum()) == 187
    ):

        print(
            "PASS: LBXGLU matches CDC."
        )

    else:

        print(
            "CHECK: LBXGLU counts differ."
        )


# ------------------------------------------------------------
# LBDGLUSI
# CDC:
# 2.537 to 30.397
# valid 3169
# missing 187
# ------------------------------------------------------------

if "LBDGLUSI" in df.columns:

    col = df["LBDGLUSI"]

    valid_range = (
        col.between(
            2.537,
            30.397
        )
    )

    print("\nLBDGLUSI")

    if (
        int(valid_range.sum()) == 3169
        and int(col.isna().sum()) == 187
    ):

        print(
            "PASS: LBDGLUSI matches CDC."
        )

    else:

        print(
            "CHECK: LBDGLUSI counts differ."
        )


# ------------------------------------------------------------
# LBXCPSI
#
# CDC:
# regular range 0.068 to 5.112 = 3131
# below detection = 0.021          = 6
# missing                          = 219
# ------------------------------------------------------------

if "LBXCPSI" in df.columns:

    col = df["LBXCPSI"]

    regular = int(
        col.between(
            0.068,
            5.112
        ).sum()
    )

    below_detection = int(
        col.eq(0.021).sum()
    )

    missing = int(
        col.isna().sum()
    )

    print("\nLBXCPSI")

    print(
        "Regular range:",
        regular
    )

    print(
        "Below detection 0.021:",
        below_detection
    )

    print(
        "Missing:",
        missing
    )

    if (
        regular == 3131
        and below_detection == 6
        and missing == 219
    ):

        print(
            "PASS: LBXCPSI matches CDC."
        )

    else:

        print(
            "CHECK: LBXCPSI counts differ."
        )


# ------------------------------------------------------------
# LBXIN
#
# CDC:
# regular range 1 to 205.69 = 3120
# below detection 0.71       = 16
# missing                    = 220
# ------------------------------------------------------------

if "LBXIN" in df.columns:

    col = df["LBXIN"]

    regular = int(
        col.between(
            1,
            205.69
        ).sum()
    )

    below_detection = int(
        col.eq(0.71).sum()
    )

    missing = int(
        col.isna().sum()
    )

    print("\nLBXIN")

    print(
        "Regular range:",
        regular
    )

    print(
        "Below detection 0.71:",
        below_detection
    )

    print(
        "Missing:",
        missing
    )

    if (
        regular == 3120
        and below_detection == 16
        and missing == 220
    ):

        print(
            "PASS: LBXIN matches CDC."
        )

    else:

        print(
            "CHECK: LBXIN counts differ."
        )


# ------------------------------------------------------------
# LBDINSI
#
# CDC:
# regular range 6 to 1234.14 = 3120
# below detection 4.26        = 16
# missing                     = 220
# ------------------------------------------------------------

if "LBDINSI" in df.columns:

    col = df["LBDINSI"]

    regular = int(
        col.between(
            6,
            1234.14
        ).sum()
    )

    below_detection = int(
        col.eq(4.26).sum()
    )

    missing = int(
        col.isna().sum()
    )

    print("\nLBDINSI")

    print(
        "Regular range:",
        regular
    )

    print(
        "Below detection 4.26:",
        below_detection
    )

    print(
        "Missing:",
        missing
    )

    if (
        regular == 3120
        and below_detection == 16
        and missing == 220
    ):

        print(
            "PASS: LBDINSI matches CDC."
        )

    else:

        print(
            "CHECK: LBDINSI counts differ."
        )


# ============================================================
# 9. FINAL TINY VALUE CHECK
# ============================================================

remaining_tiny = int(
    df.select_dtypes(
        include=[np.number]
    )
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print("\n--- FINAL ZERO FIDELITY ---")

if remaining_tiny == 0:

    print(
        "PASS: No tiny zero-artifact values remain."
    )

else:

    print(
        "FAIL:",
        remaining_tiny,
        "tiny values remain."
    )


# ============================================================
# 10. EXPORT CSV
# ============================================================

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 11. CSV ROUND-TRIP CHECK
# ============================================================

check_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


print("\n--- CSV ROUND-TRIP CHECK ---")


# SEQN should have no .0 in actual CSV text

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    header = f.readline()

    seqn_dot_zero = False

    for line in f:

        seqn = line.split(",", 1)[0]

        if seqn.endswith(".0"):

            seqn_dot_zero = True
            break


if not seqn_dot_zero:

    print(
        "PASS: SEQN exported without .0"
    )

else:

    print(
        "FAIL: SEQN contains .0"
    )


# Compare every lab measurement after CSV export

for col in LAB_MEASUREMENTS:

    if col in df.columns:

        same = np.allclose(
            df[col].to_numpy(),
            check_df[col].to_numpy(),
            equal_nan=True,
            rtol=0,
            atol=0
        )

        if same:

            print(
                f"PASS: {col} preserved exactly."
            )

        else:

            print(
                f"FAIL: {col} changed during CSV export."
            )


print("\n" + "=" * 70)
print("L10AM_C CONVERSION COMPLETE")
print("=" * 70)

NHANES 2003-2004 L10AM_C XPT -> CSV
Rows: 3,356
Columns: 7
Variables: ['SEQN', 'WTSAF2YR', 'LBXGLU', 'LBDGLUSI', 'LBXCPSI', 'LBXIN', 'LBDINSI']

--- BEFORE CORRECTION ---
True numeric zeros: 0
Tiny values: 426

Tiny values by variable:
WTSAF2YR    426
dtype: int64

WTSAF2YR values to restore to zero: 426

--- SEQN ---
Missing: 0
Duplicates: 0
dtype: Int64
LBXGLU: dtype=float64, min=45.7, max=547.6, missing=187
LBDGLUSI: dtype=float64, min=2.537, max=30.397, missing=187
LBXCPSI: dtype=float64, min=0.021, max=5.112, missing=219
LBXIN: dtype=float64, min=0.71, max=205.69, missing=220
LBDINSI: dtype=float64, min=4.26, max=1234.14, missing=220

--- CDC CODEBOOK CHECK ---
PASS: Row count = 3,356

WTSAF2YR
Zero count: 426
Missing: 0
Minimum: 0.0
Maximum: 355659.48
PASS: WTSAF2YR within CDC range.

LBXGLU
Valid: 3169
Missing: 187
PASS: LBXGLU matches CDC.

LBDGLUSI
PASS: LBDGLUSI matches CDC.

LBXCPSI
Regular range: 3131
Below detection 0.021: 6
Missing: 219
PASS: LBXCPSI matches CDC.

LBXIN
R